In [1]:
import pandas as pd

base_dados = [
    "../data/raw/1semestre/Base de dados paradas dash de departamento - Geral 2026 .01.xlsx",
    "../data/raw/base_atual/Base de Dados Paradas.xlsx",
]

partes = [pd.read_excel(a, sheet_name="PARADAS") for a in base_dados]
par = pd.concat(partes, ignore_index=True)

for coluna in ["Data inicial","Data final"]:
    par[coluna] = pd.to_datetime(par[coluna], format="mixed",dayfirst=True)

par["dur_min"] = pd.to_timedelta(par["Duração"].astype(str)).dt.total_seconds() / 60
par = par.drop(columns=["Duração", "Operador"])

par.to_parquet("../data/processed/paradas_bruto.parquet")

In [ ]:
import pandas as pd

base_dados = [
    "../data/raw/1semestre/Base de dados paradas dash de departamento - Geral 2026 .01.xlsx",
    "../data/raw/base_atual/Base de Dados Paradas.xlsx",
]

partes = [pd.read_excel(a, sheet_name="PARADAS") for a in base_dados]
par = pd.concat(partes, ignore_index=True)

print(par["Data inicial"].map(type).value_counts())

As colunas de data vêm misturando data e texto. Foi necessário converter antes de usar.

In [16]:
import pandas as pd
par = pd.read_parquet("../data/processed/paradas_bruto.parquet")

print("Quantas linhas e colunas?")
print("linhas: ", par.shape[0],"\nColunas: ", par.shape[1])
print("\n")
print("Tipo de cada coluna:")
par.info()
print("\n")
print("Período coberto:")
print(par["Data inicial"].min(), "até", par["Data inicial"].max())
print("\n")
print("Quantas máquinas (estações) diferentes?")
print("Estações diferentes:", par["Estação"].nunique())
print("\n")
print("Quantos eventos têm a MC 68 e a MC 63?")
mc68 = par[par["Estação"].str.startswith("MC 68")]
mc63 = par[par["Estação"].str.startswith("MC 63")]
print("MC 68:", len(mc68), "MC 63:", len(mc63)) ## maquinas modelo escolhidas para estudo inicial
print("\n")
print("Horas paradas por tipo de evento na MC 68 (top 10):")
top68 = mc68.groupby("Evento")["dur_min"].sum().sort_values(ascending=False) / 60
print(top68.head(10))
print("\n")
print("Horas paradas por tipo de evento na MC 63 (top 10):")
top63 = mc63.groupby("Evento")["dur_min"].sum().sort_values(ascending=False) / 60
print(top63.head(10))
print("\n")

print("Que % do tempo parado é \"Parada não Categorizada\"?")
total = mc68["dur_min"].sum()
nao_cat = mc68[mc68["Evento"].str.contains("não Categorizada", case=False)]["dur_min"].sum()
print(f"MC 68: {100 * nao_cat / total:.1f}%")
print(f"MC 63: {100 * mc63[mc63['Evento'].str.contains('não Categorizada', case=False)]['dur_min'].sum() / mc63['dur_min'].sum():.1f}%")

print("\n")
print("Linhas duplicadas (mesma estação, início, evento e duração)")
dup = par.duplicated(subset=["Estação", "Data inicial", "Evento", "dur_min"]).sum()
print("Duplicadas:", dup)

print("\n")
print("Durações estranhas")
print(par["dur_min"].describe())
print("Zero:", (par["dur_min"] == 0).sum())
print("Mais de 8 h:", (par["dur_min"] > 480).sum())

fim_calculado = par["Data inicial"] + pd.to_timedelta(par["dur_min"], unit="min")
diferenca = (par["Data final"] - fim_calculado).abs()
print("Linhas onde não bate (dif > 1 min):", (diferenca > pd.Timedelta(minutes=1)).sum())

print("\n")
par[diferenca > pd.Timedelta(minutes=1)][["Data inicial", "Data final", "dur_min"]].head(10)


Quantas linhas e colunas?
linhas:  1225472 
Colunas:  5


Tipo de cada coluna:
<class 'pandas.DataFrame'>
RangeIndex: 1225472 entries, 0 to 1225471
Data columns (total 5 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Estação       1225472 non-null  str           
 1   Evento        1225472 non-null  str           
 2   Data inicial  1225472 non-null  datetime64[us]
 3   Data final    1225472 non-null  datetime64[us]
 4   dur_min       1225472 non-null  float64       
dtypes: datetime64[us](2), float64(1), str(2)
memory usage: 84.7 MB


Período coberto:
2026-01-03 00:01:00 até 2026-10-13 23:59:00


Quantas máquinas (estações) diferentes?
Estações diferentes: 93


Quantos eventos têm a MC 68 e a MC 63?
MC 68: 33193 MC 63: 27349


Horas paradas por tipo de evento na MC 68 (top 10):
Evento
Setup Corte                        225.421111
Parada Programada para Lanche      222.296944
Parada desconhecida                177.

,Data inicial,Data final,dur_min
107297,2026-02-02 23:35:00,2026-03-02,24.200000
107314,2026-02-02 23:42:00,2026-03-02,17.383333
107316,2026-02-02 23:43:00,2026-03-02,16.500000
107317,2026-02-02 23:43:00,2026-03-02,16.150000
107321,2026-02-02 23:47:00,2026-03-02,12.666667
107323,2026-02-02 23:47:00,2026-03-02,12.000000
107328,2026-02-02 23:51:00,2026-03-02,8.566667
107329,2026-02-02 23:51:00,2026-03-02,8.566667
107330,2026-02-02 23:51:00,2026-03-02,8.216667
107331,2026-02-02 23:52:00,2026-03-02,7.633333
